In [ ]:
import os 
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing_extensions import TypedDict
from IPython.display import display, Image

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8
)

class State(TypedDict):
    topic: str
    story: str
    enhanced_story: str
    final_story: str

def generate_story(state:State):
    response = model.invoke(f"Write a one sentence story premise on topic {state['topic']}")
    return {"story": response.content}

def enhance_story(state:State):
    response = model.invoke(f"Enhance the story premise with valid details : {state['story']}")
    return {"enhanced_story": response.content}

def final_story(state:State):
    response = model.invoke(f"Add an unexpected twist to the story premise {state['enhanced_story']}")
    return {"final_story": response.content}

def check_conflict(state:State):
    if "?" in state["story"] or "!" in state["story"]:
        return "Fail"
    else:
        return "Pass"
    

# Graph creation
graph = StateGraph(State)

graph.add_node("Generate Story", generate_story)
graph.add_node("Enhance Story", enhance_story)
graph.add_node("Improve Story", final_story)

graph.add_edge(START, "Generate Story")
graph.add_conditional_edges("Generate Story", check_conflict, {"Fail": "Generate Story", "Pass": "Enhance Story"})
graph.add_edge("Enhance Story", "Improve Story")
graph.add_edge("Improve Story", END)

graph_builder = graph.compile()

display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
graph_builder.invoke({"topic": "rabbit and tortoise"})

## Parallization

import os 
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing_extensions import TypedDict
from IPython.display import display, Image

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8
)

class State(TypedDict):
  topic: str
  characters: str
  settings: str
  premise: str
  final_story: str

# Nodes
def generate_character(state:State):
  """Generate the characters of the story"""
  response = model.invoke(f"Create two character names and brief traits for a story about {state['topic']}")
  return {"characters": response.content}

def generate_settings(state:State):
  """Generate the settings of the story"""
  response = model.invoke(f"Generate a vivid settings for a story about {state['topic']}")
  return {"settings": response.content}

def generate_premise(state:State):
  """Generate the premise of the story"""
  response = model.invoke(f"Write a one sentence plot premise for a story about {state['topic']}")
  return {"premise": response.content}

def generate_story(state:State):
  """Generate the story"""
  response = model.invoke(f"""Write a story introduction using these elements.
  Characters: {state["characters"]}
  Settings: {state["settings"]}
  Premise: {state["premise"]}""")
  return {"final_story": response.content}


graph = StateGraph(State)

graph.add_node("Generate Characters", generate_character)
graph.add_node("Generate Settings", generate_settings)
graph.add_node("Generate Premise", generate_premise)
graph.add_node("Generate Story", generate_story)

graph.add_edge(START, "Generate Characters")
graph.add_edge(START, "Generate Settings")
graph.add_edge(START, "Generate Premise")
graph.add_edge("Generate Characters", "Generate Story")
graph.add_edge("Generate Settings", "Generate Story")
graph.add_edge("Generate Premise", "Generate Story")
graph.add_edge("Generate Story", END)

graph_builder = graph.compile()

display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
state = {"topic": "time travel"}
response = graph_builder.invoke(state)
response